# 08 RAG-LLM Generation — Clinical Insights per Patient


**Output format per patient (JSON):**
- `doctor_alert` — risk level + clinical summary for the attending physician
- `patient_precautions` — actionable steps to prevent readmission
- `follow_up_recommendations` — care team follow-up guidance

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'google-genai', '-q'], check=True)
print('All packages ready.')

In [ ]:
import json, os, time
from google import genai
from google.genai import types

# ===================================================
# CONFIGURATION — paste your Gemini key on the line below
# ===================================================
GEMINI_API_KEY = 'PASTE_YOUR_GEMINI_KEY_HERE'   # ← replace with your AIza... key

MAX_PATIENTS = 4508

BASE_DIR     = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
PROMPTS_PATH = os.path.join(BASE_DIR, 'rag_prompts_semantic.json')
OUTPUT_PATH  = os.path.join(BASE_DIR, 'llm_outputs_semantic.json')
# ===================================================

assert GEMINI_API_KEY != 'PASTE_YOUR_GEMINI_KEY_HERE', "Replace PASTE_YOUR_GEMINI_KEY_HERE with your actual Gemini API key"

# ── Load prompts ──────────────────────────────────────────────────────────────
with open(PROMPTS_PATH) as f:
    rag_prompts = json.load(f)
print(f"Loaded {len(rag_prompts)} semantic prompts.")

# ── Resume: skip already-processed hadm_ids ───────────────────────────────────
processed_ids = set()
if os.path.exists(OUTPUT_PATH):
    with open(OUTPUT_PATH) as f:
        existing = json.load(f)
    results = [r for r in existing if 'error' not in r]
    processed_ids = {r['hadm_id'] for r in results}
    print(f"Resuming — {len(processed_ids)} already done, {len(rag_prompts) - len(processed_ids)} remaining.")
else:
    results = []
    print("Starting fresh run.")

# ── System prompt — generates BOTH doctor and patient precautions ─────────────
SYSTEM_INSTRUCTION = (
    'You are an expert clinical decision support AI embedded in a hospital readmission prevention system. '
    'You will receive: (1) a current patient clinical summary with predicted 30-day readmission risk and '
    'SHAP risk drivers, and (2) similar historical cases retrieved from the patient database. '
    'Use the similar cases to enrich your clinical reasoning. '
    'Respond ONLY with valid JSON using this exact structure: '
    '{"doctor_alert": {"risk_level": "HIGH / MEDIUM / LOW", '
    '"risk_summary": "2-3 sentence clinical summary for the attending physician explaining why this patient is at risk, '
    'referencing the SHAP drivers, clinical history, and similar past cases."}, '
    '"doctor_precautions": ['
    '"Clinical precaution 1: specific medical action the physician should take, referencing labs or risk factors.", '
    '"Clinical precaution 2: specific clinical intervention, monitoring step, or diagnostic consideration.", '
    '"Clinical precaution 3: medication review, dosage adjustment, or treatment recommendation.", '
    '"Clinical precaution 4: discharge planning, specialist referral, or care coordination step."], '
    '"patient_precautions": ['
    '"Simple step 1 in plain language the patient can understand — practical daily action.", '
    '"Simple step 2 in plain language — medication adherence or diet tip.", '
    '"Simple step 3 in plain language — warning signs the patient should watch for.", '
    '"Simple step 4 in plain language — when to call the doctor or seek emergency care."], '
    '"follow_up_recommendations": "1-2 sentence follow-up care plan for the care team specifying timing and specialty."}. '
    'No markdown. No extra text. Only strictly valid JSON.'
)

# Gemini 2.5 Flash — Paid Tier 1
client = genai.Client(api_key=GEMINI_API_KEY)
MODEL  = 'gemini-2.5-flash'

# ── Processing loop ───────────────────────────────────────────────────────────
to_process = [p for p in rag_prompts[:MAX_PATIENTS] if p['hadm_id'] not in processed_ids]
print(f"Model: {MODEL}")
print(f"Processing {len(to_process)} patients...\n")

for i, item in enumerate(to_process):
    hid      = item['hadm_id']
    ctx      = item['prompt_context']
    risk_pct = float(item.get('risk_pct', 0.0))

    print(f"[{i+1}/{len(to_process)}] hadm_id: {hid} | risk: {risk_pct}%", end=' ... ')

    try:
        response = client.models.generate_content(
            model=MODEL,
            contents=ctx,
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_INSTRUCTION,
                response_mime_type='application/json',
                temperature=0.3
            )
        )
        parsed = json.loads(response.text)

        results.append({
            'hadm_id':                   hid,
            'predicted_risk_pct':        round(risk_pct, 1),
            'risk_level':                parsed.get('doctor_alert', {}).get('risk_level', ''),
            'doctor_alert':              parsed.get('doctor_alert', {}),
            'doctor_precautions':        parsed.get('doctor_precautions', []),
            'patient_precautions':       parsed.get('patient_precautions', []),
            'follow_up_recommendations': parsed.get('follow_up_recommendations', '')
        })
        print('OK')

    except Exception as e:
        err = str(e)
        print(f'ERROR: {err[:80]}')
        if '429' in err or 'quota' in err.lower() or 'rate' in err.lower():
            print('  Rate limit — waiting 60s then retrying...')
            time.sleep(60)
            try:
                response = client.models.generate_content(
                    model=MODEL,
                    contents=ctx,
                    config=types.GenerateContentConfig(
                        system_instruction=SYSTEM_INSTRUCTION,
                        response_mime_type='application/json',
                        temperature=0.3
                    )
                )
                parsed = json.loads(response.text)
                results.append({
                    'hadm_id':                   hid,
                    'predicted_risk_pct':        round(risk_pct, 1),
                    'risk_level':                parsed.get('doctor_alert', {}).get('risk_level', ''),
                    'doctor_alert':              parsed.get('doctor_alert', {}),
                    'doctor_precautions':        parsed.get('doctor_precautions', []),
                    'patient_precautions':       parsed.get('patient_precautions', []),
                    'follow_up_recommendations': parsed.get('follow_up_recommendations', '')
                })
                print('  Retry OK')
            except Exception as e2:
                results.append({'hadm_id': hid, 'predicted_risk_pct': round(risk_pct, 1), 'error': str(e2)})
                print(f'  Retry failed: {str(e2)[:60]}')
        else:
            results.append({'hadm_id': hid, 'predicted_risk_pct': round(risk_pct, 1), 'error': err})

    # Save after every patient
    with open(OUTPUT_PATH, 'w') as f:
        json.dump(results, f, indent=2)

    if i < len(to_process) - 1:
        time.sleep(0.1)

successful = [r for r in results if 'error' not in r]
print(f"\nDone. {len(successful)}/{len(results)} successful → {os.path.basename(OUTPUT_PATH)}")

In [ ]:
# Load and display results
BASE_DIR    = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
OUTPUT_PATH = os.path.join(BASE_DIR, 'llm_outputs_semantic.json')

with open(OUTPUT_PATH) as f:
    all_results = json.load(f)

successful = [r for r in all_results if 'error' not in r]
failed     = [r for r in all_results if 'error' in r]
print(f"Total: {len(all_results)} | Successful: {len(successful)} | Failed: {len(failed)}")
print()

RISK_LABEL = {'HIGH': 'CRITICAL', 'MEDIUM': 'MONITOR', 'LOW': 'STABLE'}

for r in successful:
    level = str(r.get('risk_level', 'UNKNOWN')).upper().strip()
    label = RISK_LABEL.get(level, level)
    da    = r.get('doctor_alert', {})

    print('=' * 65)
    print(f"PATIENT {r['hadm_id']}  |  Risk: {r['predicted_risk_pct']}%  |  {level} ({label})")
    print('=' * 65)

    print('\n[DOCTOR ALERT]')
    print(' ', da.get('risk_summary', 'N/A'))

    print('\n[PATIENT PRECAUTIONS]')
    for idx, p in enumerate(r.get('patient_precautions', []), 1):
        print(f'  {idx}. {p}')

    print('\n[FOLLOW-UP RECOMMENDATIONS]')
    print(' ', r.get('follow_up_recommendations', 'N/A'))
    print()

if failed:
    print(f"\n--- {len(failed)} FAILED patients ---")
    for r in failed[:5]:
        print(f"  hadm_id {r['hadm_id']}: {r['error'][:80]}")